# Golden questions – expected answers
Answers computed from the metric views before setting up Genie. This is the answer key for the accuracy test. Reporting month: August 2026; churn window: June–August 2026.

**Q1 (Lookup)** What was MAU in August 2026?

In [0]:
SELECT mau FROM workspace.scribd_exercise.metrics_mau
WHERE month_start = '2026-08-01'
AND business_id = 'All businesses'; 

mau
401


**Q2 (Breakdown)** How many active paid subscribers did each business have in August 2026?

In [0]:
SELECT business_id, active_paid_subscribers FROM workspace.scribd_exercise.metrics_active_subscribers
WHERE month_end = '2026-08-31'
ORDER BY business_id;

business_id,active_paid_subscribers
All businesses,285
BIZ_1,92
BIZ_2,108
BIZ_3,99


**Q3 (Lookup + split)** What was MRR in August 2026, and how much came from lifetime deals?

In [0]:
SELECT mrr_usd
, mrr_lifetime_usd
, ROUND(mrr_lifetime_usd / mrr_usd, 4) AS pct_lifetime
FROM workspace.scribd_exercise.metrics_mrr
WHERE month_end = '2026-08-31'
AND business_id = 'All businesses';

mrr_usd,mrr_lifetime_usd,pct_lifetime
3956.01,959.08,0.2424


**Q4 (Breakdown)** What was the churn rate for June–August 2026, by billing cycle?

In [0]:
SELECT billing_cycle, churn_rate, churn_rate_ci_low, churn_rate_ci_high
FROM workspace.scribd_exercise.metrics_churn
WHERE window_end = '2026-08-31'
  AND business_id = 'All businesses'
ORDER BY churn_rate DESC;

billing_cycle,churn_rate,churn_rate_ci_low,churn_rate_ci_high
monthly,0.2716,0.1867,0.3772
All billing cycles,0.1324,0.098,0.1765
annual,0.0976,0.0567,0.1628
lifetime_deal_fake,0.0482,0.0189,0.1175


**Q5 (Comparison)** Which business has the highest churn?

In [0]:
SELECT business_id, churn_rate, churn_rate_ci_low, churn_rate_ci_high
FROM workspace.scribd_exercise.metrics_churn
WHERE window_end = DATE'2026-08-31'
  AND billing_cycle = 'All billing cycles'
ORDER BY churn_rate DESC;

business_id,churn_rate,churn_rate_ci_low,churn_rate_ci_high
BIZ_1,0.1573,0.0961,0.2469
All businesses,0.1324,0.098,0.1765
BIZ_2,0.1296,0.0788,0.2059
BIZ_3,0.1111,0.0615,0.1926


**Q6 (Lookup)** What share of engagement goes to new content?

In [0]:
SELECT *
FROM workspace.scribd_exercise.metrics_content_age
WHERE month_start = '2026-08-01'
  AND business_id = 'All businesses';

month_start,business_id,age_band,band_order,content_events,engagement_share,live_items,catalog_share,engagement_per_item
2026-08-01,All businesses,New (0-90 days),1,176,0.2924,22,0.0769,8.0
2026-08-01,All businesses,Recent (91-365 days),2,197,0.3272,65,0.2273,3.03
2026-08-01,All businesses,Older (366+ days),3,229,0.3804,199,0.6958,1.15


**Q7 (Lookup, vague wording)** How good is discovery? Expected: answered with catalog reach.

In [0]:
SELECT *
FROM workspace.scribd_exercise.metrics_catalog_reach
WHERE month_start = DATE'2026-08-01'
  AND business_id = 'All businesses'
  AND dimension IN ('total', 'content age');

month_start,business_id,dimension,dimension_value,live_items,reached_items,catalog_reach,catalog_reach_ci_low,catalog_reach_ci_high
2026-08-01,All businesses,content age,Recent (91-365 days),65,62,0.9538,0.8729,0.9842
2026-08-01,All businesses,total,All,286,216,0.7552,0.7022,0.8015
2026-08-01,All businesses,content age,Older (366+ days),199,132,0.6633,0.5951,0.7253
2026-08-01,All businesses,content age,New (0-90 days),22,22,1.0,0.8513,1.0


**Q8 (Trap, lookup)** What was the churn rate for July 2026? Expected: no monthly churn exists; the closest answer is the 3-month window ending July 31 (May–July), clearly labeled.

In [0]:
SELECT window_end, churn_rate, churn_rate_ci_low, churn_rate_ci_high
FROM workspace.scribd_exercise.metrics_churn
WHERE window_end = '2026-07-31'
  AND business_id = 'All businesses'
  AND billing_cycle = 'All billing cycles';

window_end,churn_rate,churn_rate_ci_low,churn_rate_ci_high
2026-07-31,0.1423,0.1063,0.188


**Q9 (Trap, comparison)** How much has churn increased? Expected: check the direction instead of assuming an increase; compare windows that don't overlap (March–May vs. June–August); say whether the confidence intervals overlap.

In [0]:
SELECT window_end, churn_rate, churn_rate_ci_low, churn_rate_ci_high
FROM workspace.scribd_exercise.metrics_churn
WHERE window_end IN (DATE'2026-05-31', DATE'2026-08-31')
  AND business_id = 'All businesses'
  AND billing_cycle = 'All billing cycles'
ORDER BY window_end;

window_end,churn_rate,churn_rate_ci_low,churn_rate_ci_high
2026-05-31,0.1085,0.0762,0.1524
2026-08-31,0.1324,0.098,0.1765


**Q10 (Trap, lookup)** What was MRR in October 2026? Expected: no number; the metric views stop at August 2026 (#3).

In [0]:
SELECT MAX(month_end) AS latest_month_end,
       SUM(CASE WHEN month_end >= '2026-10-01' THEN 1 ELSE 0 END) AS october_rows
FROM workspace.scribd_exercise.metrics_mrr;

latest_month_end,october_rows
2026-08-31,0


**Q11 (Trap, lookup)** What's the average session time? Expected: no number; it isn't an official metric (placeholder durations double the average, #25). The query shows that no metric view has a session column.

In [0]:
SELECT table_name, column_name
FROM workspace.information_schema.columns
WHERE table_schema = 'scribd_exercise'
  AND table_name LIKE 'metrics_%'
  AND column_name LIKE '%session%';

table_name,column_name


**Q12 (Trap, lookup)** How many users do we have? Expected: ask which one is meant, or name the definition used.

In [0]:
SELECT 'MAU (August 2026)' AS definition, mau AS value
FROM workspace.scribd_exercise.metrics_mau
WHERE month_start = DATE'2026-08-01' AND business_id = 'All businesses'
UNION ALL
SELECT 'Active paid subscribers (August 31, 2026)', active_paid_subscribers
FROM workspace.scribd_exercise.metrics_active_subscribers
WHERE month_end = DATE'2026-08-31' AND business_id = 'All businesses';SELECT 'MAU (August 2026)' AS definition, mau AS value
FROM workspace.scribd_exercise.metrics_mau
WHERE month_start = DATE'2026-08-01' AND business_id = 'All businesses'
UNION ALL
SELECT 'Active paid subscribers (August 31, 2026)', active_paid_subscribers
FROM workspace.scribd_exercise.metrics_active_subscribers
WHERE month_end = DATE'2026-08-31' AND business_id = 'All businesses';

definition,value
MAU (August 2026),401
"Active paid subscribers (August 31, 2026)",285


In [0]:
SELECT window_start,
       window_end,
       churn_rate,
       churn_rate_ci_low,
       churn_rate_ci_high
FROM workspace.scribd_exercise.metrics_churn
WHERE window_end = DATE'2026-05-31'
  AND business_id = 'All businesses'
  AND billing_cycle = 'All billing cycles';

window_start,window_end,churn_rate,churn_rate_ci_low,churn_rate_ci_high
2026-03-01,2026-05-31,0.1085,0.0762,0.1524


## Held-out questions
Same rules as the golden questions, in wording that doesn't appear in the Genie instructions. Used to check that Genie learned the rules, not the examples.

**H1 (Trap).** What was churn in the first quarter of 2026? Expected: a 3-month window (January–March 2026), labeled; no monthly rate.

In [0]:
SELECT window_start, window_end, churn_rate, churn_rate_ci_low, churn_rate_ci_high
FROM workspace.scribd_exercise.metrics_churn
WHERE window_end = DATE'2026-03-31'
  AND business_id = 'All businesses'
  AND billing_cycle = 'All billing cycles';

window_start,window_end,churn_rate,churn_rate_ci_low,churn_rate_ci_high
2026-01-01,2026-03-31,0.0658,0.0409,0.1043


**H2 (Trap).** What's our average revenue per paying customer? Expected: not an official metric. The query shows that no metric view has an ARPPU or average-revenue column.

In [0]:
SELECT table_name, column_name
FROM workspace.information_schema.columns
WHERE table_schema = 'scribd_exercise'
  AND table_name LIKE 'metrics_%'
  AND (column_name LIKE '%arppu%' OR column_name LIKE '%arpu%' OR column_name LIKE '%avg%revenue%');

table_name,column_name


**H3 (Trap).** How many people used the product in September 2026? Expected: no number; the MAU view stops at August 2026.

In [0]:
SELECT MAX(month_start) AS latest_month
FROM workspace.scribd_exercise.metrics_mau;

latest_month
2026-08-01


**H4 (Lookup, held-out wording).** How many customers do we have? Expected: active paid subscribers as of August 31, 2026, with the upper bound, naming the definition.

In [0]:
SELECT active_paid_subscribers, upper_bound_paid_subscribers
FROM workspace.scribd_exercise.metrics_active_subscribers
WHERE month_end = DATE'2026-08-31'
  AND business_id = 'All businesses';

active_paid_subscribers,upper_bound_paid_subscribers
285,314


**H5 (Lookup, held-out wording).** Is our backlist getting seen? Expected: catalog reach for older content compared with newer content, August 2026.

In [0]:
SELECT dimension_value, live_items, reached_items, catalog_reach, catalog_reach_ci_low, catalog_reach_ci_high
FROM workspace.scribd_exercise.metrics_catalog_reach
WHERE month_start = DATE'2026-08-01'
  AND business_id = 'All businesses'
  AND dimension = 'content age';

dimension_value,live_items,reached_items,catalog_reach,catalog_reach_ci_low,catalog_reach_ci_high
Recent (91-365 days),65,62,0.9538,0.8729,0.9842
New (0-90 days),22,22,1.0,0.8513,1.0
Older (366+ days),199,132,0.6633,0.5951,0.7253
